# Enterprise Text-to-SQL — QLoRA Fine-Tuning (Qwen3-8B)

**Author:** Garvit Audichya ([GitHub](https://github.com/garvit-010/Voice-Text-to-SQL-FineTuning) | [HuggingFace](https://huggingface.co/garvit-010/qwen3-8b-text2sql-qlora-v3))

**Pipeline Overview:**
1. **Environment & GPU Setup:** Verifies Kaggle Nvidia T4 GPU (16 GB VRAM).
2. **Dataset:** Loads enterprise Text-to-SQL SFT dataset (2,175 train / 459 validation).
3. **QLoRA Fine-Tuning:** 4-bit NF4 quantization, LoRA r=16, alpha=32 on Qwen3-8B.
4. **Supervision:** Supervised only on target SQL tokens (lm_head loss computed on ~45 SQL tokens, not prompt).
5. **Evaluation:** Generates greedy predictions on the held-out 453-question test set.
6. **Export:** Packages final adapter and training metrics for deployment.

**Execution Time:** ~7 hours on single Nvidia T4 (Save & Run All / Commit mode).


## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers>=5.0 peft>=0.20 bitsandbytes>=0.50 accelerate>=1.0 trl>=1.0 datasets sqlglot huggingface-hub
print("Dependencies installed!")

## Step 2: Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "NO GPU! Go to Settings -> Accelerator -> GPU T4 x2"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
print(f"Compute: {props.major}.{props.minor}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print("\nGPU is ready!")

## Step 3: Locate Dataset

The dataset should be uploaded as a Kaggle Dataset and added to this notebook as input.

In [ ]:
import os, json
from pathlib import Path

# Auto-detect the dataset from Kaggle input
kaggle_input = Path("/kaggle/input")
train_file = val_file = script_file = None

if kaggle_input.is_dir():
    for candidate in sorted(kaggle_input.iterdir()):
        t = next(candidate.rglob("train.jsonl"), None)
        v = next(candidate.rglob("validation.jsonl"), None)
        s = next(candidate.rglob("train_qlora.py"), None)
        if t and v:
            train_file, val_file = str(t), str(v)
        if s:
            script_file = str(s)

assert train_file, "Dataset not found! Add your uploaded dataset as Kaggle input."

# Count examples
n_train = sum(1 for l in open(train_file) if l.strip())
n_val = sum(1 for l in open(val_file) if l.strip())
print(f"Train file: {train_file} ({n_train} examples)")
print(f"Val file:   {val_file} ({n_val} examples)")
if script_file:
    print(f"Script:     {script_file}")
print("\nDataset ready!")

## Step 4: Run Training

This uses the `train_qlora.py` script which handles:
- Loading Qwen3-8B in 4-bit NF4
- Attaching LoRA adapters (r=16, alpha=32)
- Completion-only loss (only trains on SQL, not the prompt)
- Checkpointing every 50 steps

**This takes ~7 hours on a T4. Let it run.**

In [ ]:
# Use 1 GPU only (DataParallel on 2xT4 causes issues with label-only loss)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

OUTPUT_DIR = "/kaggle/working/qwen3-8b-text2sql-qlora-v3"

# Run training with 1 epoch, gradient accumulation 16 for effective batch size 16
!python {script_file} \
    --train-file {train_file} \
    --val-file {val_file} \
    --output-dir {OUTPUT_DIR} \
    --epochs 1 \
    --grad-accum 16

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

## Step 5: Verify Adapter Was Saved

In [ ]:
adapter_dir = Path(OUTPUT_DIR) / "final_adapter"
assert adapter_dir.exists(), f"Adapter not found at {adapter_dir}"

files = list(adapter_dir.iterdir())
total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1024 / 1024
print(f"Adapter saved at: {adapter_dir}")
print(f"Files: {len(files)}")
print(f"Total size: {total_mb:.1f} MB")
for f in sorted(files):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

# Show training metrics
metrics_file = Path(OUTPUT_DIR) / "training_metrics.json"
if metrics_file.exists():
    metrics = json.loads(metrics_file.read_text())
    print(f"\nTraining time: {metrics.get('train_runtime_h', '?')} hours")
    print(f"Final train loss: {metrics.get('train_loss', '?')}")
    if 'final_eval' in metrics:
        print(f"Final eval loss: {metrics['final_eval'].get('eval_loss', '?')}")

## Step 6: Generate Predictions on Test Set

Load the eval pack and generate SQL for all 453 test questions.

In [ ]:
import hashlib, time, re
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Find eval pack
eval_pack_file = None
for candidate in sorted(kaggle_input.iterdir()):
    ep = next(candidate.rglob("eval_pack.jsonl"), None)
    if ep:
        eval_pack_file = ep
        break

assert eval_pack_file, "Eval pack not found! Upload eval_pack.jsonl as Kaggle input."
eval_questions = [json.loads(l) for l in eval_pack_file.read_text().splitlines() if l.strip()]
print(f"Eval pack: {eval_pack_file} ({len(eval_questions)} questions)")

# Load prompt and schema from the eval pack directory
prompt_file = eval_pack_file.parent / "prompt_module.py"
schema_file = eval_pack_file.parent / "schema_context.txt"

# Import prompt module
import importlib.util
spec = importlib.util.spec_from_file_location("prompt_module", str(prompt_file))
prompt_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prompt_mod)

SCHEMA = schema_file.read_text(encoding="utf-8")
SCHEMA_FP = hashlib.sha256(SCHEMA.encode()).hexdigest()[:16]
PROMPT_FP = prompt_mod.prompt_fingerprint()

print(f"Prompt fingerprint: {PROMPT_FP}")
print(f"Schema fingerprint: {SCHEMA_FP}")
print(f"Schema length: {len(SCHEMA)} chars")

In [ ]:
# Load model + adapter
BASE_MODEL = "Qwen/Qwen3-8B"
REVISION = "b968826d9c46"

print("Loading base model in 4-bit...")
compute_dtype = torch.bfloat16 if torch.cuda.get_device_properties(0).major >= 8 else torch.float16

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=REVISION,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype),
    device_map={"":0},
    low_cpu_mem_usage=True,
)

print("Loading QLoRA adapter...")
model = PeftModel.from_pretrained(base, str(adapter_dir)).eval()

tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded! VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
def extract_sql(raw: str) -> str:
    """Extract SQL from model output."""
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.S | re.I)
    text = re.sub(r"<think>.*", "", text, flags=re.S | re.I).strip()
    fenced = re.search(r"```(?:sql|postgresql)?\s*(.*?)```", text, re.S | re.I)
    if fenced:
        text = fenced.group(1).strip()
    match = re.search(r"\b(WITH|SELECT)\b", text, re.I)
    if match:
        text = text[match.start():]
    if ";" in text:
        text = text.split(";", 1)[0]
    return " ".join(text.split()).strip()


def generate_one(question: str) -> tuple[str, str, float]:
    """Generate SQL for a single question. Returns (raw, sql, ms)."""
    messages = prompt_mod.build_messages(question, SCHEMA)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False)  # CRITICAL: must be False for this adapter
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    
    started = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=512, do_sample=False,
                            pad_token_id=tokenizer.pad_token_id)
    elapsed = (time.perf_counter() - started) * 1000
    
    raw = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    sql = extract_sql(raw)
    return raw, sql, elapsed

# Quick test
test_raw, test_sql, test_ms = generate_one("How many customers are there?")
print(f"Test: {test_sql}")
print(f"Time: {test_ms:.0f} ms")
print("\nGeneration working!")

In [ ]:
# Generate predictions for ALL test questions
output_file = Path("/kaggle/working/predictions_v3.jsonl")
predictions = []
total = len(eval_questions)

print(f"Generating predictions for {total} questions...")
started_all = time.perf_counter()

for i, q in enumerate(eval_questions):
    raw, sql, ms = generate_one(q["question"])
    pred = {
        "id": q["id"],
        "question": q["question"],
        "raw_output": raw,
        "predicted_sql": sql,
        "generation_ms": round(ms, 1),
        "prompt_fingerprint": PROMPT_FP,
        "schema_fingerprint": SCHEMA_FP,
        "prompt_format": "v2_no_think",
        "adapter": str(adapter_dir),
    }
    predictions.append(pred)
    
    # Append immediately (checkpoint)
    with open(output_file, "a") as f:
        f.write(json.dumps(pred) + "\n")
    
    if (i+1) % 25 == 0 or (i+1) == total:
        elapsed = (time.perf_counter() - started_all)
        rate = (i+1) / elapsed
        eta = (total - i - 1) / rate if rate > 0 else 0
        print(f"  [{i+1}/{total}] {elapsed/60:.1f} min elapsed, ~{eta/60:.1f} min remaining")

total_time = (time.perf_counter() - started_all)
print(f"\nDone! {total} predictions in {total_time/60:.1f} minutes")
print(f"Output: {output_file}")

## Step 7: Package for Download

Everything you need to score locally and deploy.

In [ ]:
import shutil

# Create a package directory
pkg = Path("/kaggle/working/v3_results")
pkg.mkdir(exist_ok=True)

# Copy adapter
shutil.copytree(adapter_dir, pkg / "final_adapter", dirs_exist_ok=True)

# Copy training artifacts
for f in ["training_config.json", "training_metrics.json"]:
    src = Path(OUTPUT_DIR) / f
    if src.exists():
        shutil.copy2(src, pkg / f)

# Copy predictions
shutil.copy2(output_file, pkg / "predictions_v3.jsonl")

# Create archive
archive = shutil.make_archive("/kaggle/working/v3_results", "zip", pkg)
print(f"Archive: {archive} ({Path(archive).stat().st_size / 1024 / 1024:.1f} MB)")
print("\nDownload this zip file and extract it in your project!")
print("Then run scoring locally:")
print("  python scripts/score_finetuned.py --version v2 --predictions predictions_v3.jsonl")